In [ ]:
# 00 — 언록 성능 확인
GPU 텐서코어 언록 전/후 성능을 이 노트북에서 직접 재현합니다.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)
print("언록 상태: journalctl -k -b | grep gpu_unlock 으로 확인 (unlocked)")

In [ ]:
# BF16 GEMM (언록 대상 경로) — 언록 서버 기준 ~130 TFLOPS
import time, statistics, torch
a = torch.randn(8192, 8192, device="cuda", dtype=torch.bfloat16)
b = torch.randn(8192, 8192, device="cuda", dtype=torch.bfloat16)
for _ in range(3): torch.matmul(a, b)
torch.cuda.synchronize()
ts = []
for _ in range(7):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    torch.matmul(a, b); torch.cuda.synchronize()
    ts.append(time.perf_counter() - t0)
tflops = 2 * 8192**3 / statistics.median(ts) / 1e12
print(f"BF16 GEMM: {tflops:.1f} TFLOPS (잠긴 카드 기준 71)")

In [ ]:
# FlashAttention 2 처리량 (fwd) — 잠긴 상한 71 TFLOPS 대비
from flash_attn import flash_attn_func
import torch, time, statistics
B, H, S, D = 1, 32, 2048, 128
q = torch.randn(B, S, H, D, device="cuda", dtype=torch.bfloat16)
k = torch.randn(B, S, H, D, device="cuda", dtype=torch.bfloat16)
v = torch.randn(B, S, H, D, device="cuda", dtype=torch.bfloat16)
def step(): flash_attn_func(q, k, v, causal=True); torch.cuda.synchronize()
for _ in range(3): step()
ts = []
for _ in range(7):
    t0 = time.perf_counter(); step(); ts.append(time.perf_counter() - t0)
tflops = 4 * B * H * S * S * D / statistics.median(ts) / 1e12
print(f"FlashAttention fwd: {tflops:.1f} TFLOPS")